# Module import

In [ ]:
from netCDF4 import Dataset                             
import numpy as np            

import matplotlib.pyplot as plt           
import matplotlib.colors as mcolors              
import cartopy.crs as ccrs                              
import cartopy.feature as cfeature     
import cartopy.io.shapereader as shpreader

import sys
import torch 
import warnings
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta

import os
from scipy.ndimage import gaussian_filter
from matplotlib.colors import ListedColormap

In [ ]:
sys.path.append("/home/users/mendrika/Object-Based-LSTMConv/notebooks/model/training")
from pancast import Core2MapModel

# Choose nowcast origin and lead time (in hour)

In [ ]:
year = "2024"
month = "09"
day = "30"
hour = "13"
minute = "00"

# Ground truth 

In [ ]:
def update_hour(date_dict, hours_to_add, minutes_to_add):
    """
    Add hours and minutes to a datetime dictionary and return the updated dict and a generated file path.

    Args:
        date_dict     (dict): Keys: 'year', 'month', 'day', 'hour', 'minute' as strings, e.g. "01", "23"
        hours_to_add   (int): Number of hours to add.
        minutes_to_add (int): Number of minutes to add.

    Returns:
        tuple:
            - dict: Updated datetime dictionary with all fields as zero-padded strings.
            - str: File path in the format YYYY/MM/YYYYMMDDHHMM.nc
    """
    # Parse the original time
    time_obj = datetime(
        int(date_dict["year"]),
        int(date_dict["month"]),
        int(date_dict["day"]),
        int(date_dict["hour"]),
        int(date_dict["minute"])
    )

    # Add hours
    updated = time_obj + timedelta(hours=hours_to_add, minutes=minutes_to_add)

    # Format updated dictionary
    new_date_dict = {
        "year":   f"{updated.year:04d}",
        "month":  f"{updated.month:02d}",
        "day":    f"{updated.day:02d}",
        "hour":   f"{updated.hour:02d}",
        "minute": f"{updated.minute:02d}"
    }

    # Generate file path
    file_path = f"{new_date_dict['year']}/{new_date_dict['month']}/{new_date_dict['year']}{new_date_dict['month']}{new_date_dict['day']}{new_date_dict['hour']}{new_date_dict['minute']}.nc"

    return {'time': new_date_dict, 'path': file_path}

In [ ]:
def load_wavelet_dataset(year, month, day, hour, minute, lead_time):
    
    nowcast_origin = {
        "year":   year,
        "month":  month,
        "day":    day,
        "hour":   hour,
        "minute": minute,
    }

    nowcast_lt = update_hour(nowcast_origin, hours_to_add=0, minutes_to_add=lead_time)["time"]

    path_core = f"/gws/nopw/j04/cocoon/SSA_domain/ch9_wavelet/{nowcast_lt['year']}/{nowcast_lt['month']}"
    file = f"{path_core}/{nowcast_lt['year']}{nowcast_lt['month']}{nowcast_lt['day']}{nowcast_lt['hour']}{nowcast_lt['minute']}.nc"
    return Dataset(file, mode='r')["cores"]

In [ ]:
y_min, y_max = 48, 2062
x_min, x_max = 81, 2267

In [ ]:
geodata = np.load("/gws/nopw/j04/cocoon/SSA_domain/lat_lon_2268_2080.npz")
lons = geodata["lon"][y_min:y_max+1, x_min:x_max+1]
lats = geodata["lat"][y_min:y_max+1, x_min:x_max+1]

In [ ]:
def load_zcast_input(year, month, day, hour, minute, lead_time):

    input_paths = [
        "/work/scratch-nopw2/mendrika/pancast/raw/inputs_t0",
        "/gws/nopw/j04/wiser_ewsa/mrakotomanga/pancast/raw/inputs_t0",
    ]

    fname = f"input-{year}{month}{day}_{hour}{minute}.pt"

    for base in input_paths:
        core_input = f"{base}/{fname}"
        if os.path.exists(core_input):
            return torch.load(core_input)

    raise FileNotFoundError(
        f"ZCAST input not found in any known location: {fname}"
    )

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
ENSEMBLE_DIR_T030 = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/pancast/final/checkpoints/t030min"
ENSEMBLE_DIR_T060 = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/pancast/final/checkpoints/t060min"
ENSEMBLE_DIR_T090 = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/pancast/final/checkpoints/t090min"
ENSEMBLE_DIR_T120 = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/pancast/final/checkpoints/t120min"

In [ ]:
def load_models(ensemble_dir):
    models = []
    for seed in sorted(os.listdir(ensemble_dir)):
        ckpt = os.path.join(ensemble_dir, seed, "lr7e-05/best-pancast.ckpt")
        if os.path.exists(ckpt):
            model = Core2MapModel.load_from_checkpoint(ckpt, map_location=DEVICE)
            model.eval().to(DEVICE)
            models.append(model)
            print(f"Loaded {ckpt}")
    return models

In [ ]:
def ensemble_predict(models, x):
    preds = []
    with torch.no_grad():
        for model in models:
            pred = torch.sigmoid(model(x)).squeeze(0).squeeze(0)
            preds.append(pred)
    preds = torch.stack(preds)
    mean_pred = preds.mean(dim=0)
    var_pred  = preds.var(dim=0)
    return mean_pred, var_pred

In [ ]:
models_t030 = load_models(ENSEMBLE_DIR_T030)
models_t060 = load_models(ENSEMBLE_DIR_T060)
models_t090 = load_models(ENSEMBLE_DIR_T090)
models_t120 = load_models(ENSEMBLE_DIR_T120)

In [ ]:
SCALER_PATH = "/home/users/mendrika/Object-Based-LSTMConv/outputs/scaler-africa/scaler_realcores_online.pt"

MASK_COL_INDEX = 12
COLS_TO_SCALE = range(4, 12)

# load scaler
scaler = torch.load(SCALER_PATH, weights_only=False)
mean = np.asarray(scaler["mean"])
scale = np.asarray(scaler["scale"])

In [ ]:
zcast_input = load_zcast_input(year, month, day, hour, minute, 0)

input_tensor = zcast_input["input_tensor"].clone().unsqueeze(0)

In [ ]:
try:
    # load one instance
    zcast_input = load_zcast_input(year, month, day, hour, minute, 0)

    input_tensor = zcast_input["input_tensor"].clone().unsqueeze(0)

    # remove batch dim for scaling
    X = input_tensor[0]

    # convert to numpy
    X_np = X.numpy()

    flat = X_np.reshape(-1, X_np.shape[-1])

    flat[:, COLS_TO_SCALE] = (flat[:, COLS_TO_SCALE] - mean) / scale

    X_scaled = torch.tensor(flat.reshape(X_np.shape), dtype=torch.float32)

    input_scaled = X_scaled.unsqueeze(0)

    mean_pred_t030, _ = ensemble_predict(models_t030, input_scaled.to(DEVICE))
    mean_pred_t060, _ = ensemble_predict(models_t060, input_scaled.to(DEVICE))
    mean_pred_t090, _ = ensemble_predict(models_t090, input_scaled.to(DEVICE))
    mean_pred_t120, _ = ensemble_predict(models_t120, input_scaled.to(DEVICE))

except:
    print("No data")

In [ ]:
preds = {
    30: mean_pred_t030,
    60: mean_pred_t060,
    90: mean_pred_t090,
    120: mean_pred_t120
}

In [ ]:
def gamma_boost(pred, gamma=2.0):
    pred = pred.copy()
    pred = np.clip(pred, 0.0, 1.0)
    return pred ** gamma


def rescale_after_threshold(pred, floor=0.05, eps=1e-6):
    pred = pred.copy()
    pred[pred < floor] = 0.0

    max_val = pred.max()
    if max_val > eps:
        pred = pred / max_val
    return pred

In [ ]:
timestamp = f"{year}{month}{day}_{hour}{minute}"

for lead, pred in preds.items():

    pred = rescale_after_threshold(pred.numpy(), floor=0.1)
    pred = gamma_boost(pred, gamma=0.85)

    pred = np.ma.masked_where(pred < 0.05, pred)

    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(alpha=0)

    fig, ax = plt.subplots(
        figsize=(9,9),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    extent = (-18, 55, -35, 40)
    ax.set_extent(extent, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="black", linewidth=0.6, zorder=5)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.9, zorder=6)

    gl = ax.gridlines(
        draw_labels=True,
        linestyle="--",
        linewidth=0.25,
        color="gray",
        alpha=0.4,
        xlocs=np.arange(-20, 60, 20),
        ylocs=np.arange(-40, 30, 20),
    )

    gl.top_labels = True
    gl.bottom_labels = True
    gl.left_labels = True
    gl.right_labels = True

    gl.xlabel_style = {"size":11}
    gl.ylabel_style = {"size":11}

    p = ax.pcolormesh(
        lons,
        lats,
        pred,
        cmap=cmap,
        vmin=0.05,
        vmax=1,
        shading="nearest",
        zorder=2,
    )

    cb = fig.colorbar(
        p,
        ax=ax,
        orientation="horizontal",
        pad=0.07,       # increase distance from axis labels
        fraction=0.045
    )

    cb.set_label("Thunderstorm probability", fontsize=11)

    archive = "/home/users/mendrika/PANCAST/outputs/archive"

    filename = f"{archive}/{timestamp}_{lead}.png"

    plt.savefig(filename, dpi=600, bbox_inches="tight")
    plt.close()

In [ ]:
fig, ax = plt.subplots(
    figsize=(9,9),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

extent = (-18, 55, -35, 40)
ax.set_extent(extent, crs=ccrs.PlateCarree())

ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="black", linewidth=0.6, zorder=5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.9, zorder=6)

gl = ax.gridlines(
    draw_labels=True,
    linestyle="--",
    linewidth=0.25,
    color="gray",
    alpha=0.4,
    xlocs=np.arange(-20, 60, 20),
    ylocs=np.arange(-40, 30, 20),
)

gl.top_labels = True
gl.bottom_labels = True
gl.left_labels = True
gl.right_labels = True

gl.xlabel_style = {"size":11}
gl.ylabel_style = {"size":11}

# create dummy mappable for the colorbar only
norm = plt.Normalize(vmin=0.05, vmax=1)
sm = plt.cm.ScalarMappable(norm=norm, cmap="YlOrRd")
sm.set_array([])

cb = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    pad=0.07,
    fraction=0.045
)

cb.set_label("Thunderstorm probability", fontsize=11)

placeholder = "/home/users/mendrika/PANCAST/outputs/no_nowcast.png"

plt.savefig(placeholder, dpi=300, bbox_inches="tight")
plt.close()